In [1]:
import cv2
import json
import pickle
import glob
import math
import random
import matplotlib.pyplot as plt
from collections import defaultdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast, GradScaler
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset, DataLoader, random_split
from typing import List, Tuple
from tqdm import tqdm
from models import StackedHourglassCBAM
from utils import softargmax_2d
random.seed(20) # 10, 11, 12
device = torch.device("cuda:0")
#device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
with open("pickle/x_ray_2/test.pkl", "rb") as f:
    test_data = pickle.load(f)

In [ ]:
len(test_data)

In [4]:
def crop_image(image, point, crop_percent):
    h, w = image.shape[:2]
    x, y = point  # Unpack the coordinates from the point tuple
    
    # Calculate the side length based on a percentage of the shortest dimension
    side = int(min(w, h) * crop_percent)
    half = side // 2

    # Determine crop boundaries (Clamped to stay inside image frames)
    # This logic ensures the square stays 'side' length even near edges
    left = int(max(0, min(x - half, w - side)))
    top = int(max(0, min(y - half, h - side)))

    # Perform the crop using NumPy slicing
    cropped = image[top:top+side, left:left+side]
    
    return cropped, np.array([left, top])

In [5]:
def generate_heatmap(size_hw: Tuple[int, int], center_xy: Tuple[float, float], sigma: float = 2.0):
    """Create a single 2D gaussian heatmap (H,W) with center (x,y) in pixel coords."""
    W, H = size_hw[1], size_hw[0]
    y = torch.arange(H, dtype=torch.float32)
    x = torch.arange(W, dtype=torch.float32)
    yy, xx = torch.meshgrid(y, x, indexing="ij")
    cx, cy = center_xy
    hm = torch.exp(-((xx - cx) ** 2 + (yy - cy) ** 2) / (2 * sigma ** 2))
    return hm

In [6]:
h_model = StackedHourglassCBAM(num_keypoints=1, num_stacks=2, depth=4, channels=256, in_ch=1).to(device)
h_model.load_state_dict(torch.load('saved/x_ray_2/hourglass_cbam[hip].pth', weights_only=True))
h_model.to(device)
h_model.eval()

k_model = StackedHourglassCBAM(num_keypoints=6, num_stacks=2, depth=4, channels=256, in_ch=1).to(device) # 12
k_model.load_state_dict(torch.load('saved/x_ray_2/hourglass_cbam_low[knee].pth', weights_only=True))
k_model.to(device)
k_model.eval()

a_model = StackedHourglassCBAM(num_keypoints=1, num_stacks=2, depth=4, channels=256, in_ch=1).to(device)
a_model.load_state_dict(torch.load('saved/x_ray_2/hourglass_cbam[ankle].pth', weights_only=True))
a_model.to(device)
a_model.eval()

roi_model = StackedHourglassCBAM(num_keypoints=4, num_stacks=2, depth=4, channels=256, in_ch=1).to(device)
roi_model.load_state_dict(torch.load('saved/x_ray_2/hourglass_cbam[roi].pth', weights_only=True))
roi_model.to(device)
roi_model.eval()
print("Loaded!")

Loaded!


In [7]:
def calc_angle(joint_vec, mech_vec):
    dot_product = np.dot(joint_vec, mech_vec)
    norm_joint = np.linalg.norm(joint_vec)
    norm_mech = np.linalg.norm(mech_vec)

    # Use a small epsilon to avoid division by zero
    denominator = norm_joint * norm_mech
    if denominator < 1e-8:
        return 0.0  # Or np.nan, depending on how you want to handle errors
    
    cos_theta = dot_product / denominator
    angle_rad = np.arccos(np.clip(cos_theta, -1.0, 1.0))
    return np.degrees(angle_rad)

In [8]:
LABELS = (
    "R1", "R2", "R3", "R4", # crops
    "RM1", "RM2",
    "RSL1", "RSM1",
    "RSLT1", "RSMT1"
)

In [9]:
indices = []
angle_errs = []

for i in range(len(test_data)):
    # ROI
    image_path, keypoints = test_data[i]
    #image_array = np.fromfile(image_path, dtype=np.uint8)
    image_array = np.fromfile(f"/{image_path}", dtype=np.uint8)
    image = cv2.imdecode(image_array, cv2.IMREAD_GRAYSCALE)
    H, W = image.shape

    resized_image = cv2.resize(image, (256, 512))
    image_tensor = torch.from_numpy(resized_image).float().unsqueeze(0) / 255.0

    keypoints = [keypoints[l] for l in LABELS]
    keypoints = np.array([(x, H-y) for x, y in keypoints], dtype=np.float32)

    hgt_kp = keypoints[0].copy()
    agt_kp = keypoints[3].copy()
    kgt_kps = keypoints[4:].copy()
    with torch.no_grad():
        x = image_tensor.unsqueeze(0).to(device) # add batch_dim
        outs = roi_model(x)
    p_hmps = outs[-1]
    p_kps = softargmax_2d(p_hmps, beta=100.0)
    p_kps = p_kps.squeeze().cpu().numpy()

    p_kps[:, 0] *= 256 / 64
    p_kps[:, 1] *= 512 / 128

    #plt.figure(figsize=(10, 10))
    #plt.imshow(resized_image, cmap='gray')
    #plt.scatter(p_kps[:, 0], p_kps[:, 1], c='red', s=20)
    #plt.axis('off')
    #plt.show()

    # HIP
    h_cc = p_kps[0].copy()
    h_cc[0] *= W/256
    h_cc[1] *= H/512
    
    # KNEE
    k_cc = np.sum(p_kps[1:3], axis=0) / 2
    k_cc[0] *= W/256
    k_cc[1] *= H/512
    
    # ANKLE
    a_cc = p_kps[3].copy()
    a_cc[0] *= W/256
    a_cc[1] *= H/512

    # ROI [VISUALIZATION]
    #plt.figure(figsize=(12, 12))
    #plt.imshow(image, cmap='gray')
    #plt.scatter(h_cc[0], h_cc[1], c='red', marker='o', s=20) # HIP
    #plt.scatter(k_cc[0], k_cc[1], c='red', marker='o', s=20) # KNEE
    #plt.scatter(a_cc[0], a_cc[1], c='red', marker='o', s=20) # ANKLE
    #plt.axis('off')
    #plt.show()

    h_cimg, h_shift = crop_image(image, h_cc, crop_percent=0.25)
    k_cimg, k_shift = crop_image(image, k_cc, crop_percent=0.3)
    a_cimg, a_shift = crop_image(image, a_cc, crop_percent=0.25)

    # CROPPED HIP, KNEE AND ANKLE [VISUALIZATION]
    #fig, axes = plt.subplots(1, 3, figsize=(12, 12))

    #axes[0].imshow(h_cimg, cmap='gray')
    #axes[0].axis('off')
    
    #axes[1].imshow(k_cimg, cmap='gray')
    #axes[1].axis('off')
    
    #axes[2].imshow(a_cimg, cmap='gray')
    #axes[2].axis('off')
    #plt.show()

    # HIP
    h_img = cv2.resize(h_cimg, (384, 384))
    h_img = torch.from_numpy(h_img).float().unsqueeze(0) / 255.0
    
    with torch.no_grad():
            h_inp = h_img.unsqueeze(0).to(device)
            h_out = h_model(h_inp)
    
    hp_hmp = h_out[-1]
    
    h_img = h_img.squeeze().numpy()
    
    hp_kp = softargmax_2d(hp_hmp, beta=100.0)
    hp_kp = hp_kp.squeeze().cpu().numpy()
    hp_kp[0] *= 384 / 96
    hp_kp[1] *= 384 / 96
    
    hp_kp[0] *= h_cimg.shape[0] / 384
    hp_kp[1] *= h_cimg.shape[1] / 384
    hp_kp += h_shift
    
    # KNEE
    k_img = cv2.resize(k_cimg, (512, 512))
    k_img = torch.from_numpy(k_img).float().unsqueeze(0) / 255.0
    
    with torch.no_grad():
        k_inp = k_img.unsqueeze(0).to(device)
        k_outs = k_model(k_inp)
    
    kp_hmps = k_outs[-1]
    
    k_img = k_img.squeeze().numpy()
    
    kp_kps = softargmax_2d(kp_hmps, beta=100.0)
    kp_kps = kp_kps.squeeze().cpu().numpy()
    kp_kps[:, 0] *= 512 / 128
    kp_kps[:, 1] *= 512 / 128
    
    kp_kps[:, 0] *= k_cimg.shape[0] / 512
    kp_kps[:, 1] *= k_cimg.shape[1] / 512
    kp_kps += k_shift
    
    # # ANKLE
    a_img = cv2.resize(a_cimg, (256, 256))
    a_img = torch.from_numpy(a_img).float().unsqueeze(0) / 255.0
    
    with torch.no_grad():
        a_inp = a_img.unsqueeze(0).to(device)
        a_out = a_model(a_inp)
    
    ap_hmp = a_out[-1]
    
    a_img = a_img.squeeze().numpy()
    
    ap_kp = softargmax_2d(ap_hmp, beta=100.0)
    ap_kp = ap_kp.squeeze().cpu().numpy()
    ap_kp[0] *= 256 / 64
    ap_kp[1] *= 256 / 64
    
    ap_kp[0] *= a_cimg.shape[0] / 256
    ap_kp[1] *= a_cimg.shape[0] / 256
    ap_kp += a_shift


    p_joint_vec = kp_kps[5] - kp_kps[4]
    p_mech_vec = ap_kp - kp_kps[1]
    
    gt_joint_vec = kgt_kps[5] - kgt_kps[4]
    gt_mech_vec = agt_kp - kgt_kps[1]

    p_angle = calc_angle(p_joint_vec, p_mech_vec)
    gt_angle = calc_angle(gt_joint_vec, gt_mech_vec)

    angle_err = abs(p_angle - gt_angle)
    angle_errs.append(angle_err)


    #print(f"MPTA: {p_angle}")
    #plt.figure(figsize=(16, 16))
    #plt.imshow(image, cmap='gray')
    #plt.scatter(kp_kps[5, 0], kp_kps[5, 1], c='red', marker='o', s=30)
    #plt.scatter(kp_kps[4, 0], kp_kps[4, 1], c='yellow', marker='o', s=30)
    # plt.plot([kp_kps[5, 0], kp_kps[4, 0]], [kp_kps[5, 1], kp_kps[4, 1]], c='yellow')
    
    # plt.scatter(ap_kp[0], ap_kp[1], c='lime', marker='o', s=30)
    # plt.scatter(kp_kps[1, 0], kp_kps[1, 1], c='blue', marker='o', s=30)

    # plt.plot([ap_kp[0], kp_kps[1, 0]], [ap_kp[1], kp_kps[1, 1]], c='lightblue')
    
    # plt.axis()
    # plt.show()

In [ ]:
print(f"mean: {np.mean(np.array(angle_errs))}")
print(f"std: {np.std(np.array(angle_errs))}")